# Amharic Braille Detection - Demo

This notebook demonstrates the complete Amharic Braille detection pipeline.

In [ ]:
import sys
sys.path.insert(0, 'src')

from braille_preprocessing import BraillePreprocessor
from braille_segmentation import BrailleSegmenter
from braille_classifier import BrailleClassifier
from main import BrailleDetector
from utils import display_image, visualize_pipeline_stages

import cv2
import matplotlib.pyplot as plt
import numpy as np

## 1. Load and Display Input Image

In [ ]:
# Load a sample image
image_path = './raw_dataset/img5.jpg'
image = cv2.imread(image_path)

# Display
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Original Braille Image')
plt.axis('off')
plt.show()

## 2. Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = BraillePreprocessor()

# Preprocess the image
preprocessed = preprocessor.preprocess(image_path)

# Visualize stages
stages = {
    'Original': preprocessed['original'],
    'Grayscale': preprocessed['gray'],
    'Enhanced': preprocessed['enhanced'],
    'Deskewed': preprocessed['deskewed'],
    'Binary': preprocessed['binary']
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, img) in enumerate(stages.items()):
    if len(img.shape) == 3:
        axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(name)
    axes[idx].axis('off')

axes[-1].axis('off')  # Hide last unused subplot
plt.tight_layout()
plt.show()

print(f"Binary image shape: {preprocessed['binary'].shape}")

## 3. Segmentation

In [ ]:
# Initialize segmenter
segmenter = BrailleSegmenter(
    min_dot_size=5,
    dot_spacing_threshold=20,
    letter_spacing_threshold=20
)

# Segment the image
segmentation = segmenter.segment(preprocessed['binary'], preprocessed['deskewed'])

print(f"Detected {len(segmentation['dot_contours'])} dots")
print(f"Grouped into {len(segmentation['rows'])} rows")
print(f"Extracted {len(segmentation['letter_images'])} characters")

# Visualize segmentation
visualization = segmenter.visualize_segmentation(preprocessed['deskewed'], segmentation)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(visualization, cv2.COLOR_BGR2RGB))
plt.title('Segmented Characters')
plt.axis('off')
plt.show()

## 4. Display Individual Characters

In [ ]:
# Display extracted letters
n_letters = len(segmentation['letter_images'])
if n_letters > 0:
    cols = min(5, n_letters)
    rows = (n_letters + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
    if n_letters == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_letters > 1 else [axes]
    
    for idx, letter_info in enumerate(segmentation['letter_images']):
        axes[idx].imshow(letter_info['image'], cmap='gray')
        axes[idx].set_title(f"Character {idx+1}")
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(n_letters, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No characters detected")

## 5. Classification (Requires Trained Model)

Note: The classifier requires a trained model. For demonstration purposes, the model will make predictions based on its initialized (untrained) state.

In [ ]:
# Initialize classifier
classifier = BrailleClassifier()

print("\nNote: Model is untrained. Predictions will be random.")
print("To get accurate results, train the model with labeled Braille data.\n")

# Classify each character
if segmentation['letter_images']:
    predictions = []
    for idx, letter_info in enumerate(segmentation['letter_images']):
        char, confidence = classifier.predict(letter_info['image'])
        predictions.append((char, confidence))
        print(f"Character {idx+1}: '{char}' (confidence: {confidence:.2%})")
    
    # Create detected text
    detected_text = ''.join([char for char, conf in predictions])
    avg_confidence = np.mean([conf for char, conf in predictions])
    
    print(f"\nDetected Text: {detected_text}")
    print(f"Average Confidence: {avg_confidence:.2%}")
else:
    print("No characters to classify")

## 6. Complete Pipeline using BrailleDetector

In [ ]:
# Initialize complete detector
detector = BrailleDetector()

# Process image
result = detector.process_image(
    image_path,
    save_intermediates=True,
    output_dir='output/demo'
)

print(f"\nDetected {result['num_characters']} characters")

if result['predictions']:
    detected_text = ''.join([char for char, conf in result['predictions']])
    print(f"Detected Text: {detected_text}")
    
    # Show confidence for each character
    print("\nCharacter-by-character results:")
    for idx, (char, conf) in enumerate(result['predictions'], 1):
        print(f"  {idx}. '{char}' - {conf:.2%}")

# Display final visualization
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(result['visualization'], cv2.COLOR_BGR2RGB))
plt.title('Final Detection Result')
plt.axis('off')
plt.show()

print("\nIntermediate results saved to output/demo/")

## 7. Batch Processing Example

In [ ]:
# Process all images in raw_dataset
batch_results = detector.process_batch(
    input_dir='raw_dataset',
    output_dir='output/batch',
    save_intermediates=False
)

print(f"\nProcessed {len(batch_results)} images")
print("\nSummary:")
for filename, result in batch_results.items():
    if 'error' in result:
        print(f"  {filename}: Error - {result['error']}")
    else:
        text = ''.join([char for char, conf in result.get('predictions', [])])
        print(f"  {filename}: {len(result.get('predictions', []))} chars - '{text}'")

## Next Steps

1. **Collect Training Data**: Gather labeled Amharic Braille images
2. **Train the Model**: Use the `classifier.train()` method with your dataset
3. **Evaluate**: Test accuracy on a validation set
4. **Fine-tune**: Adjust parameters for better performance
5. **Deploy**: Save the trained model and use it for production